<a href="https://colab.research.google.com/github/ALiao18/ALiao18/blob/main/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pprint import pprint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
import re

In [ ]:
link = '/content/drive/MyDrive/S25/Langone/Febrile_Seizure_Prediction/Data/n330_final.xlsx'
df = pd.read_excel(link)

In [ ]:
age_mean = df['age_sudc'].mean()
age_std = df['age_sudc'].std()

In [ ]:
# IDs to exclude (Grouped for clarity)
exclude_ids = [
    15, 28, 50, 125, 271, 283, 368,                   # Epilepsy/Afebrile
    87, 112, 124, 140, 150, 157, 200, 205, 216, 309, 315, 335 # Unknown position
]

df_clean = df[
    (~df['sudcrrc_id'].isin(exclude_ids)) &
    (df['sudcrrc_id'].notna())].copy()

print(f"Data cleaned. Final sample size: N = {len(df_clean)}")

Data cleaned. Final sample size: N = 317


In [ ]:
RED_COLS = [ #125 cols
    # free text / metadata / administrative
    'notes',
    'fi_not_completed',
    'clinical_medical_records_review_complete',
    'opine_mod_orig_mec',
    'Unnamed: 464', 'c',
    'opine_fcod_orig_path',

    # pregnancy / birth / newborn screening
    'pregnancy_pmh_details',
    'newborn_hear_test_performed',
    'newborn_hear_test_results',
    'newborn_screen_test_performed',
    'newborn_screen_test_results',
    'sudc_childs_delivery___998',

    # neurological conditions
    # 'neuro_condit___1', no longer in new dataset
    'neuro_condit___2', 'neuro_condit___3',
    'neuro_condit___7', 'neuro_condit___8', 'neuro_condit___9',
    'neuro_condit___10', 'neuro_condit___998', 'neuro_condit___99',

    'febrile_sz___1', 'f+EA:KHebrile_sz___8',
    'epilepsy___1', 'epilepsy___8',
    'autism___1', 'autism___8',
    'intellectual_disabilities___1', 'intellectual_disabilities___8',
    'neuro_other___1', 'neuro_other___8',

    'seizures_were_categorized___3',
    'seizures_were_categorized___4',
    'seizures_were_categorized___5',

    # cardiac conditions
    'cardiaccondit___999', 'cardiaccondit___998', 'cardiaccondit___99',

    'cardiac_arrhythmia___1', 'cardiac_arrhythmia___8',
    'sudden_cardiac_death___1', 'sudden_cardiac_death___8',
    'syncope___1',
    'cardiac_other___1', 'cardiac_other___2', 'cardiac_other___3',
    'cardiac_other___4', 'cardiac_other___5', 'cardiac_other___6',
    'cardiac_other___7', 'cardiac_other___8',

    # respiratory / sleep conditions
    'respcondit___3', 'respcondit___6', 'respcondit___998', 'respcondit___99',
    'asthma___1', 'asthma___8',
    'sleep_apnea___1', 'sleep_apnea___8',

    # psychiatric conditions
    'psychcondit___2', 'psychcondit___99', 'psychcondit___998',

    # metabolic / genetic conditions
    'metabolic_disease___1', 'metabolic_disease___2',
    'metabolic_disease___3', 'metabolic_disease___4',
    'metabolic_disease___5', 'metabolic_disease___6',
    'metabolic_disease___7', 'metabolic_disease___8',

    'genetic_hx___3', 'genetic_hx___99',

    # family history / medical tolerance
    'vaccine_tolerance_fam',

    # medications / fertility / medical exposure
    'fertility_meds___99',
    'term_meds_fa',

    # activity / symptom history
    'recent_actvity_hx___8',
    'recent_actvity_hx___9',

    'last_48hr_sxs___1',
    'last_48hr_sxs___8',
    'last_48hr_sxs___10',

    # exercise history
    'sudc_exer_hx___0', 'sudc_exer_hx___1', 'sudc_exer_hx___2',
    'sudc_exer_hx___3', 'sudc_exer_hx___4', 'sudc_exer_hx___5',
    'sudc_exer_hx___6', 'sudc_exer_hx___7', 'sudc_exer_hx___8',
    'sudc_exer_hx___998', 'sudc_exer_hx___99', 'sudc_exer_hx___999',

    # terminal triggers
    'term_triggers___1', 'term_triggers___2', 'term_triggers___3',
    'term_triggers___4', 'term_triggers___5', 'term_triggers___6',
    'term_triggers___7', 'term_triggers___8', 'term_triggers___9',
    'term_triggers___99', 'term_triggers___998',

    # death scene / body position
    'child_body_position_fi___1', 'child_body_position_fi___3',
    'child_body_position_fi___4', 'child_body_position_fi___5',
    'child_body_position_fi___6', 'child_body_position_fi___7',
    'child_body_position_fi___8',

    'child_face_position_fi___2', 'child_face_position_fi___3',
    'child_face_position_fi___4', 'child_face_position_fi___5',
    'child_face_position_fi___6', 'child_face_position_fi___7',

    # special needs
    'special_needs',

    # cause of death / classification
    'sudden_explained_death___1', 'sudden_explained_death___8',
    'sudden_unexplained_death___1', 'sudden_unexplained_death___8',

    #'cdc_line_1a_fpc',
    'fpc_dc_1bcd',
    'fpc_dc_line2',
    'dc_line_1a_fpc',

    'intrinsic_factors_usd',
    'extrinsic_factors_usd',
    'original_vs_fpc_cod',

    # remove crib_sleep
    'start_sleep_location_fi'
]

DEM_SOCIO_COLS = [ # 21
    'bmi', 'head_circumference', 'weight_length', 'dad_educ_level', 'mom_educ_level',
    'mother_marital_status', 'family_income_fa', 'type_residence', 'vaccine_tolerance_fam',
    'sudc_child_ethnicity___10', 'sudc_child_ethnicity___11', 'sudc_child_ethnicity___2',
    'sudc_child_ethnicity___3', 'sudc_child_ethnicity___4', 'sudc_child_ethnicity___5',
    'sudc_child_ethnicity___6', 'sudc_child_ethnicity___7', 'sudc_child_ethnicity___8',
    'sudc_child_ethnicity___9', 'desc_favesleep', 'sudc_child_ethnicity___1',
]

regex_pattern = r'99$|998$|999$|_0$'
SENTINEL_COLS = df.filter(regex=regex_pattern).columns.tolist() # 29

ZERO_VAR_COLS = ['pmh_pregnancy___1', 'pmh_pregnancy___5', 'pmh_pregnancy___12',
       'pmh_pregnancy___37', 'in_hospital_newborn_hx___19',
       'in_hospital_newborn_hx___20', 'rheu_inflam___3', 'rheu_inflam___4',
       'rheu_inflam___5', 'rheu_inflam___6', 'rheu_inflam___7',
       'rheu_inflam___8', 'rheu_inflam___9', 'rheu_inflam___10',
       'cardiaccondit___3', 'cardiaccondit___4', 'cardiaccondit___5',
       'cardiaccondit___6', 'psychcondit___3', 'genetic_hx___2',
       'sudc_health_hx___2', 'sudc_health_hx___14', 'autism___2', 'autism___3',
       'sudden_cardiac_death___2', 'sudden_cardiac_death___3',
       'sudden_cardiac_death___5', 'syncope___4', 'syncope___5', 'syncope___8',
       'sudden_explained_death___5', 'sudden_unexplained_death___2',
       'sudden_unexplained_death___3', 'sudden_unexplained_death___5',
       'bedsharing_details_fi___7', 'bedsharing_details_fi___8',
       'bedsharing_details_fi___9']

COLLINEAR_COLS = [
    'labor_details___8',   # corr=0.986 with sudc_childs_delivery___1 and 0.978 with sudc_childs_delivery___2
                           # duplicate Cesarean indicator; sudc_childs_delivery_* variables clearer; delivery type duplication
    'psychcondit___5',      # corr=1.000 with seizure
    'sudc_health_hx___1',    # redundant encoding of seizure
    'tachypnea',           # corr=1.000 with tachycardia; either could be kept; arbitrarily keeping tachycardia
    'difficulty_regulation_temp',  # corr=0.999 with require_cpr and 0.999 with vent
    'vent'                 # corr=1.000 with require_cpr, redundant with CPR indicator
]

HIGH_MISSINGNESS_COLS = [ # >35% missingness
    'sleephrspernight'
]

OTHER_DROP_COL = [
    'child_body_position_fi___2', # redundant, already have face position info
    'parasomnias___8', 'parasomnias___9', # represent None and I dont recall
    'bedsharing_details_fi___11', # represent NA
    'last_48hr_sxs___14', # represent None
    'sudden_unexplained_death___4', # represent full sibling #1, however #2 is dropped. Keep only second degree parents for interpretability
    'sudden_explained_death___4', # same as above
    'sudden_cardiac_death___4', # same as above
    'pmh_pregnancy___998', 'pmh_pregnancy___41', 'pmh_pregnancy___99', # sentinal
    'labor_details___12', 'labor_details___13', # sentinal
    'in_hospital_newborn_hx___25', 'in_hospital_newborn_hx___26', 'in_hospital_newborn_hx___998', 'in_hospital_newborn_hx___999', # sentinal
    'rheu_inflam___0', 'rheu_inflam___998', 'rheu_inflam___999', # sentinal
    'neuro_condit___11', 'neuro_condit___13', # sentinal
    'genetic_hx___998', # sentinal
    'fertility_meds___4', # sentinal
    'sudc_health_hx___0', 'sudc_health_hx___99', 'sudc_health_hx___998', # sentinal
    'found_location_fi',
    'pmh_pregnancy___29', # indicates gestational other Rx drugs
    'cardiaccondit___10', # indicates cardiovascular other
    'sudc_childs_delivery___1', # indicates vaginal delivery, already covered
    'sudc_childs_delivery___2', # indicates c section, already covered
    'last_48hr_sxs___13', # indicates injury or other conditions not mentioned above
    'autism___4', 'cardiac_arrhythmia___4', # full sibling 2 autism. Remove because full sibling 1 dropped due to 0 var
    'cigarettes', # redundant with stronger signal gestational cigarette smoking from medical records
    'death_sleep_period_fi', # this dataset already guarantees all deaths are sleep related.
    'parasomnias___1', # laura said to drop
]

In [ ]:
DROP_COLS = set(set(SENTINEL_COLS) |
                set(DEM_SOCIO_COLS) |
                set(RED_COLS) |
                set(ZERO_VAR_COLS) |
                set(COLLINEAR_COLS) |
                set(HIGH_MISSINGNESS_COLS) |
                set(OTHER_DROP_COL)
                )
len(DROP_COLS)

229

In [ ]:
# rename respiratory condition and labor detail columns
rename_map = {
    'respcondit___1': 'h/o_asthma_in_life',
    'respcondit___2': 'h/o_sleep_apnea_in_life',
    'respcondit___4': 'h/o_reactive_airway_disease_in_life',
    'respcondit___5': 'h/o_RSV_in_life',
    'labor_details___1': 'spontaneous_labor',
    'labor_details___2': 'utilized_drugs_to_initiate_labor',
    'labor_details___3': 'utilized_drugs_strengthen_contractions',
    'labor_details___4': 'fetal_distress_during_labor',
    'labor_details___5': 'meconium_staining_of_amniotic_fluid_present_labor',
    'labor_details___6': 'umbilical_cord_problems',
    'labor_details___7': 'vaginal_delivery',
    'labor_details___10': 'cesarean_delivery',
    'labor_details___11': 'fetal_distress_upon_delivery',
    'pmh_pregnancy___2': 'hypertension_during_pregnancy',
    'pmh_pregnancy___3': 'tocolytics_given_during_pregnancy',
    'pmh_pregnancy___4': 'proteinuria_during_pregnancy',
    'pmh_pregnancy___6': 'bedrest_during_pregnancy',
    'pmh_pregnancy___7': 'diabetes_req_insulin_during_pregnancy',
    'pmh_pregnancy___8': 'diabetes_not_req_insulin_during_pregnancy',
    'pmh_pregnancy___9': 'vaginal_bleeding_in_1st_trimester_during_pregnancy',
    'pmh_pregnancy___10': 'vaginal_bleeding_in_2nd_trimester_during_pregnancy',
    'pmh_pregnancy___11': 'vaginal_bleeding_in_3rd_trimester_during_pregnancy',
    'pmh_pregnancy___13': 'ciagarette_smoking_during_preganancy',
    'pmh_pregnancy___14': 'alcohol_during_pregnancy',
    'pmh_pregnancy___15': 'vitamin_during_pregnancy',
    'pmh_pregnancy___16': 'iron_supplements_during_pregnancy',
    'pmh_pregnancy___17': 'aspirin_during_pregnancy',
    'pmh_pregnancy___18': 'acetominophen_during_pregnancy',
    'pmh_pregnancy___19': 'ibuprofen_during_pregnancy',
    'pmh_pregnancy___20': 'cardiac_medications_during_pregnancy',
    'pmh_pregnancy___22': 'antiseizure_medication_during_pregnancy',
    'pmh_pregnancy___23': 'antacid_medication_during_pregnancy',
    'pmh_pregnancy___24': 'blood_pressure_medication_during_pregnancy',
    'pmh_pregnancy___26': 'antidepressants_during_pregnancy',
    'pmh_pregnancy___27': 'decongestants_during_pregnancy',
    'pmh_pregnancy___28': 'THC_during_pregnancy',
    'pmh_pregnancy___30': 'antibiotics_during_pregnancy',
    'pmh_pregnancy___31': 'maternal_infection_during_pregnancy',
    'pmh_pregnancy___32': 'hormones_during_pregnancy',
    'pmh_pregnancy___33': 'nausea_medication_during_pregnancy',
    'pmh_pregnancy___34': 'illicit_drug_use_during_pregnancy',
    'pmh_pregnancy___35': 'premature_labor_history',
    'pmh_pregnancy___38': 'third_trimester_fever',
    'pmh_pregnancy___39': 'preeclampsia_eclampsia_history',
    'parasomnias___2': '6m_prior_death_talk_during_sleep',
    'parasomnias___3': '6m_prior_death_restless_during_sleep',
    'parasomnias___4': '6m_prior_death_sleepwalked',
    'parasomnias___5': '6m_prior_death_grinded_teeth_during_sleep',
    'parasomnias___6': '6m_prior_death_awoke_screaming_sleep',
    'parasomnias___10': '6m_prior_death_awoke_sweating_sleep',
    'parasomnias___7': '6m_prior_death_alarmed_by_scary_dreams',
    'seizures_were_categorized___1': 'simple_FS(s)',
    'seizures_were_categorized___2': 'complex_FS(s)',
    'in_hospital_newborn_hx___1': 'baby_in_hospital_period_anemic',
    'in_hospital_newborn_hx___2': 'baby_in_hospital_period_req_oxygen',
    'in_hospital_newborn_hx___3': 'baby_in_hospital_period_req_CPR',
    'in_hospital_newborn_hx___4': 'baby_in_hospital_period_req_incubator',
    'in_hospital_newborn_hx___5': 'baby_in_hospital_period_req_antibiotics',
    'in_hospital_newborn_hx___6': 'baby_in_hospital_period_jaundic',
    'in_hospital_newborn_hx___7': 'baby_in_hospital_period_req_surgery',
    'in_hospital_newborn_hx___8': 'baby_in_hospital_period_infection',
    'in_hospital_newborn_hx___9': 'birth_defect',
    'in_hospital_newborn_hx___10': 'baby_in_hospital_period_tachycardia',
    'in_hospital_newborn_hx___11': 'baby_in_hospital_period_bradycardia',
    'in_hospital_newborn_hx___12': 'baby_in_hospital_period_arrhythmia',
    'in_hospital_newborn_hx___13': 'baby_in_hospital_period_apnea',
    'in_hospital_newborn_hx___14': 'baby_in_hospital_period_cyanosis',
    'in_hospital_newborn_hx___15': 'baby_in_hospital_period_excessive_vomiting',
    'in_hospital_newborn_hx___16': 'baby_in_hospital_period_resp_distress',
    'in_hospital_newborn_hx___18': 'baby_in_hospital_period_fever',
    'in_hospital_newborn_hx___21': 'baby_in_hospital_period_hypoglycemia',\
    'in_hospital_newborn_hx___22': 'baby_in_hospital_period_NICU',
    'in_hospital_newborn_hx___23': 'discharged_home_with_mom',
    'in_hospital_newborn_hx___24': 'extended hospital stay',
    'rheu_inflam___11': 'immunoglobulins_abnormal',
    'neuro_condit___4': 'child_autism',
    'neuro_condit___5': 'child_intellectual_disabilities',
    'neuro_condit___6': 'child developmental_disabilities',
    'cardiaccondit___1': 'child_cardiac_arrhythmic',
    'cardiaccondit___8': 'child_atrial_septal_defect',
    'cardiaccondit___9': 'child_ventricular_septal_defect',
    'psychcondit___1': 'child_anxiety',
    'psychcondit___4': 'child_depression',
    'psychcondit___6': 'child_ADD/ADHD',
    'genetic_hx___1': 'abnormal_chromosomes',
    'fertility_meds___1': 'fertility_meds_IVF',
    'fertility_meds___2': 'fertility_meds_IUI',
    'fertility_meds___3': 'fertility_meds_injectable_medications',
    'fertility_meds___5': 'fertility_meds_oral',
    'sudc_health_hx___3': 'sudc_child_limpness',
    'sudc_health_hx___4': 'sudc_child_became_blue',
    'sudc_health_hx___5': 'sudc_child_choking',
    'sudc_health_hx___6': 'sudc_child_developmental_delay',
    'sudc_health_hx___7': 'sudc_child_12+hrs_no_eating',
    'sudc_health_hx___8': 'sudc_child_12+hrs_vomiting',
    'sudc_health_hx___9': 'sudc_child_breathing_problems',
    'sudc_health_hx___10': 'sudc_child_heart_problems',
    'sudc_health_hx___11': 'sudc_child_sleep_disorder',
    'sudc_health_hx___12': 'sudc_child_GER',
    'sudc_health_hx___13': 'sudc_child_use_apnea_monitor_at_home',
    'sudc_health_hx___15': 'sudc_child_hearing_impairment',
    'sudc_health_hx___16': 'sudc_child_speech/language_delay',
    'sudc_health_hx___17': 'sudc_child_vision_impairment',
    'sudc_health_hx___18': 'sudc_child_wheezing',
    'sudc_health_hx___19': 'sudc_child_asthma',
    'sudc_health_hx___20': 'sudc_child_fainting',
    'recent_actvity_hx___1': '2weeks_prior_death_head_trauma',
    'recent_actvity_hx___2': '2weeks_prior_death_fly_on_plane',
    'recent_actvity_hx___3': '2weeks_prior_death_visit_ER',
    'recent_actvity_hx___4': '2weeks_prior_death_hospitalization',
    'recent_actvity_hx___5': '2weeks_prior_death_doctor_visit',
    'recent_actvity_hx___6': '2weeks_prior_death_exposure_contagious_disease',
    'recent_actvity_hx___7': '2weeks_prior_death_suffer_illness',
    'last_48hr_sxs___2': 'last_48hr_cold_symptoms',
    'last_48hr_sxs___3': 'last_48hr_lethargy',
    'last_48hr_sxs___4': 'last_48hr_crankiness',
    'last_48hr_sxs___5': 'last_48hr_excessive_crying',
    'last_48hr_sxs___6': 'last_48hr_appetite_changes',
    'last_48hr_sxs___7': 'last_48hr_vomiting',
    'last_48hr_sxs___9': 'last_48hr_fever',
    'last_48hr_sxs___11': 'last_48hr_diarrhea',
    'last_48hr_sxs___12': 'last_48hr_stool_changes',
    'bedsharing_details_fi___1': 'sharing_sleep_surface_with_another_<5yr_child',
    'bedsharing_details_fi___2': 'sharing_sleep_surface_with_another_6-10yr_child',
    'bedsharing_details_fi___3': 'sharing_sleep_surface_with_another_>10yr_child',
    'bedsharing_details_fi___4': 'sharing_sleep_surface_with_adult',
    'bedsharing_details_fi___5': 'more_than_one_person_sharing_sleep_surface'
}

In [ ]:
import pandas as pd
import numpy as np
import re

def clean_sudc_data(df, rename_map=rename_map):
    df_cleaned = df.copy()

    # --- Internal Helpers ---
    def z_standardize(series):
        if series.std() == 0 or series.isnull().all():
            return series
        return (series - series.mean()) / series.std()

    def clean_val(val, strict=False):
        s = str(val).lower()
        if any(x in s for x in ['unk', 'not', 'n/a', 'missing']):
            return 0.0 if (strict and 'none.' in s) else np.nan

        nums = re.findall(r"(\d+\.?\d*)", s)
        if not nums: return np.nan
        # If strict or only one number, return first. If range, return midpoint.
        if strict or len(nums) == 1: return float(nums[0])
        return (float(nums[0]) + float(nums[1])) / 2

    def collapse_family_history(df, prefix, suffix_range, new_col):
        cols = [f"{prefix}___{i}" for i in suffix_range]
        existing = [c for c in cols if c in df.columns]
        if existing:
            df[new_col] = (df[existing] == 1).any(axis=1).astype(int)
            df.drop(columns=existing, inplace=True)
        return df

    # --- 1. Categorical & Binary Mapping ---
    mappings = {
        'binary': ({1: 1, 2: 0, 3: np.nan, 99: np.nan, 999: np.nan},
                   ['newborn_hear_test_performed', 'newborn_screen_test_performed', 'rolled',
                    'crawled', 'favesleep', 'fallingasleep', 'stayingasleep', 'vaccines_uptodate',
                    'first_words', 'started', 'special_needs', 'term_meds_fa']),
        'milestones': ({1: 1, 2: 0, 3: 1, 4: np.nan},
                       ['milestones', 'expressive_speech', 'comprehension', 'fine_motor', 'gross_motor', 'behavior']),
        'substances': ({1: 0, 2: 0.5, 3: 1, 4: 4, 5: 30, 99: np.nan},
                       ['otc_prenatal_vitamins', 'otc_acetominophen', 'alcohol', 'caffeine_coffee_tea',
                        'cocaine', 'amphetamines', 'cannabi', 'opiods_heroin'])
    }

    for map_dict, cols in mappings.values():
        for col in [c for c in cols if c in df_cleaned.columns]:
            df_cleaned[col] = df_cleaned[col].map(map_dict)

    # Specific Binary Adjustments
    if 'gender' in df_cleaned.columns:
        df_cleaned['female'] = df_cleaned['gender'].map({1: 0, 2: 1})
        df_cleaned.drop(columns='gender', inplace=True)

    if 'allergy_hx' in df_cleaned.columns:
        df_cleaned['allergy_hx'] = df_cleaned['allergy_hx'].map({1: 1, 2: 0, 3: 1, 99: np.nan})

    # --- 2. Continuous Values & Z-Scores ---
    # Clean string percentiles/counts
    pct_cols = ['bw_percentile', 'bl_percentile', 'sick_visits_per_year']
    for col in [c for c in pct_cols if c in df_cleaned.columns]:
        df_cleaned[col] = df_cleaned[col].apply(lambda x: clean_val(x, strict=('sick' in col)))

    # Apply Z-Standardization
    z_cols = ['age_sudc', 'labor_began', 'sleephrspernight', 'gestational_age_at_birth',
              'sick_visits_per_year', 'days_illness_inyear', 'mother_age', 'father_age',
              'general_health_scale', 'bw_percentile', 'bl_percentile']

    for col in [c for c in z_cols if c in df_cleaned.columns]:
        df_cleaned[col] = z_standardize(pd.to_numeric(df_cleaned[col], errors='coerce'))

    # --- 3. Ordinal to Midpoint/Continuous ---
    ord_map = {1: 1.5, 2: 5, 3: 10, 4: 25, 5: 50, 6: 75, 7: 90, 8: 95, 9: 98.5, 10: np.nan}
    for col in ['birthweight_precent', 'birthlength_precent']:
        if col in df_cleaned.columns:
            df_cleaned[col] = df_cleaned[col].map(ord_map)

    # Miscellaneous mappings
    if 'breastfed' in df_cleaned.columns:
        df_cleaned['breastfed_3m_plus'] = df_cleaned['breastfed'].map({1: 0, 2: 0, 3: 1, 4: 1, 5: 1, 99: np.nan})
        df_cleaned.drop(columns='breastfed', inplace=True)

    time_maps = {
        'last_vacc_to_death': {1: 12, 2: 36, 3: 60, 4: 120, 5: 228, 6: 312, 7: 540, 8: 1440, 9: 2880, 99: np.nan},
        'conception_duration': {100: 3, 101: 9, 102: 15, 103: 21.5, 104: 24}
    }
    for col, tmap in time_maps.items():
        if col in df_cleaned.columns:
            df_cleaned[col] = df_cleaned[col].map(tmap)

    # --- 4. Feature Engineering (Blood & Multiples) ---
    if {'baby_blood_type', 'maternal_blood_type'}.issubset(df_cleaned.columns):
        invalid_mask = df_cleaned['baby_blood_type'].isin([5, 6]) | df_cleaned['maternal_blood_type'].isin([5, 6])
        df_cleaned['blood_mismatch'] = (df_cleaned['baby_blood_type'] != df_cleaned['maternal_blood_type']).astype(int)
        df_cleaned.loc[invalid_mask, 'blood_mismatch'] = np.nan

    if 'preg_multiple' in df_cleaned.columns:
        df_cleaned['is_multiple'] = df_cleaned['preg_multiple'].map({1: 0, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1})
        df_cleaned['birth_order_multiple'] = df_cleaned['preg_multiple'].map({2: 1, 3: 2, 4: 1, 5: 2, 6: 1, 7: 2, 8: 3}).fillna(1)
        df_cleaned.drop(columns=['preg_multiple'], inplace=True)

    # Collapse bed-sharing into single binary
    bed_sharing_cols = [f'bedsharing_details_fi___{i}' for i in range(1, 6)]
    existing_bs = [c for c in bed_sharing_cols if c in df_cleaned.columns]
    print(f"existing_bs: {existing_bs}")
    if existing_bs:
        df_cleaned['any_bed_sharing'] = (df_cleaned[existing_bs] == 1).any(axis=1).astype(int)
        df_cleaned.drop(columns=existing_bs, inplace=True)

    # --- 5. Cleaning Counts & One-Hots ---
    count_cols = ['gravida', 'para', 'miscarriage', 'abortions', 'stillbirths', 'order_preg_sudc', 'gestation']
    df_cleaned[count_cols] = df_cleaned[count_cols].replace([9, 10, 99, 998, 999], np.nan)

    # Clean One-Hots
    oh_cols = [c for c in df_cleaned.columns if '___' in c]
    df_cleaned[oh_cols] = df_cleaned[oh_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Drop low-variance one-hots (sum <= 1)
    df_cleaned.drop(columns=[c for c in oh_cols if df_cleaned[c].sum() <= 1], inplace=True)

    # if 'start_sleep_location_fi' in df_cleaned.columns:
    #     df_cleaned['sleep_crib'] = (df_cleaned['start_sleep_location_fi'] == 1).astype(int)
    #     df_cleaned.drop(columns='start_sleep_location_fi', inplace=True)

    if 'gestation' in df_cleaned.columns:
        df_cleaned['born_full_term'] = (df_cleaned['gestation'] == 1).astype(int)
        df_cleaned.drop(columns='gestation', inplace = True)

    # --- 6. Family History Aggregations ---
    fam_history_configs = [
        ('febrile_sz', range(2, 8), 'any_1st_or_2nd_degree_Fam_Hx_FS'),
        ('cardiac_arrhythmia', range(2, 8), 'any_1st_or_2nd_degree_fam_cardiac_arrhythmia'),
        ('syncope', range(2, 8), 'any_1st_or_2nd_degree_fam_syncope'),
        ('autism', range(6, 8), 'any_2nd_degree_fam_autism'),
        ('sudden_cardiac_death', range(6, 8), 'any_2nd_degree_fam_sudden_cardiac_death'),
        ('intellectual_disabilities', range(2, 8), 'any_fam_intellectual_disabilities'),
        ('neuro_other', range(2, 8), 'any_fam_other_neurological_disease'),
        ('epilepsy', range(2, 8), 'any_fam_epilepsy'),
        ('sleep_apnea', range(2, 8), 'any_fam_sleep_apnea'),
        ('asthma', range(2, 8), 'any_fam_asthma'),
        ('sudden_explained_death', range(2, 8), 'any_fam_sudden_explained_death'),
        ('sudden_unexplained_death', range(2, 8), 'any_fam_sudden_unexplained_death')
    ]

    for prefix, s_range, new_name in fam_history_configs:
        df_cleaned = collapse_family_history(df_cleaned, prefix, s_range, new_name)

    df_cleaned.rename(columns=rename_map, inplace=True)

    return df_cleaned

In [ ]:
def apply_imputation(df_cleaned):
    # 1. Identify Column Types
    # Continuous: Z-scored variables and floats
    # Categorical: Binary 0/1, One-Hots, and Rank Ordinals

    # We can use the logic from our previous cleaning patterns to separate them
    one_hot_cols = [c for c in df_cleaned.columns if '___' in c or c == 'female' or c == 'blood_mismatch']
    binary_cols = ['newborn_hear_test_performed', 'newborn_screen_test_performed',
                   'rolled', 'crawled', 'favesleep', 'fallingasleep', 'stayingasleep',
                   'vaccines_uptodate', 'first_words', 'started', 'special_needs',
                   'term_meds_fa', 'prone', 'is_multiple']

    categorical_features = list(set(one_hot_cols + binary_cols))
    # Ensure we only try to impute columns that actually exist in the dataframe
    categorical_features = [c for c in categorical_features if c in df_cleaned.columns]

    # Continuous features are everything else that is numeric
    all_numeric = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
    continuous_features = [c for c in all_numeric if c not in categorical_features]

    # 2. Initialize Imputers
    # Median is safer than Mean for clinical data with high max values
    cont_imputer = SimpleImputer(strategy='median')
    # Most_frequent (Mode) is standard for binary/one-hot flags
    cat_imputer = SimpleImputer(strategy='most_frequent')

    # 3. Apply Imputation
    df_imputed = df_cleaned.copy()

    if continuous_features:
        df_imputed[continuous_features] = cont_imputer.fit_transform(df_imputed[continuous_features])

    if categorical_features:
        df_imputed[categorical_features] = cat_imputer.fit_transform(df_imputed[categorical_features])

    return df_imputed

In [ ]:
DF_DROPPED = df_clean.drop(columns = DROP_COLS)
DF_RECODE = clean_sudc_data(DF_DROPPED)
DF_IMPUTED = apply_imputation(DF_RECODE)
dropped_cols = set(DF_DROPPED.columns) - set(DF_RECODE.columns)
added_cols = set(DF_RECODE.columns) - set(DF_DROPPED.columns)

existing_bs: ['bedsharing_details_fi___1', 'bedsharing_details_fi___2', 'bedsharing_details_fi___3', 'bedsharing_details_fi___4', 'bedsharing_details_fi___5']


/tmp/ipykernel_15846/1101030411.py:126: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_cleaned['born_full_term'] = (df_cleaned['gestation'] == 1).astype(int)
/tmp/ipykernel_15846/1101030411.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (df[existing] == 1).any(axis=1).astype(int)
/tmp/ipykernel_15846/1101030411.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd

In [ ]:
print(f"df_clean shape: {df_clean.shape}")
print(f"DF_DROPPED shape: {DF_DROPPED.shape}")
print(f"DF_RECODE shape: {DF_RECODE.shape}")
print(f"diff cols DF_DROPPED, DF_RECODE: added={list(added_cols)}, \ndropped={list(dropped_cols)}")
print(f"{len(added_cols)} added; {len(dropped_cols)} dropped")
print(f"Final shape after imputation: {DF_IMPUTED.shape}")

df_clean shape: (317, 500)
DF_DROPPED shape: (317, 271)
DF_RECODE shape: (317, 214)
diff cols DF_DROPPED, DF_RECODE: added=['hormones_during_pregnancy', 'last_48hr_excessive_crying', 'born_full_term', 'child_ventricular_septal_defect', 'iron_supplements_during_pregnancy', 'baby_in_hospital_period_tachycardia', 'meconium_staining_of_amniotic_fluid_present_labor', '2weeks_prior_death_visit_ER', 'last_48hr_appetite_changes', 'baby_in_hospital_period_anemic', 'umbilical_cord_problems', 'alcohol_during_pregnancy', '6m_prior_death_sleepwalked', 'breastfed_3m_plus', 'child_intellectual_disabilities', 'any_fam_intellectual_disabilities', 'blood_pressure_medication_during_pregnancy', 'baby_in_hospital_period_req_oxygen', 'ibuprofen_during_pregnancy', 'h/o_asthma_in_life', 'baby_in_hospital_period_infection', 'illicit_drug_use_during_pregnancy', 'proteinuria_during_pregnancy', 'premature_labor_history', 'last_48hr_fever', 'sudc_child_sleep_disorder', 'fertility_meds_IVF', 'baby_in_hospital_perio

In [ ]:
file_path = '/content/drive/MyDrive/S25/Langone/Febrile_Seizure_Prediction/Data/DF_IMPUTED.csv'

# Save the DataFrame to CSV in Google Drive
DF_IMPUTED.to_csv(file_path, index=False)